# Phase 4: Modelling (Part 1 - Baseline Random Forest with Feature Selection)
**Client:** Crédit Nationale Azur
**Objective:** Build a baseline Random Forest classifier using **SelectKBest feature selection (k=7)** to model personal loan acceptance based on customer attributes.

In this notebook, we implement the strict CRISP-DM data preparation pipeline with feature selection before training a baseline Random Forest model.

In [ ]:
# Required Base Imports
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

## 1. Load Data and Prepare Targets
We load the cleaned data from Phase 3, map the target variable to binary integers, and separate features from the target.

In [ ]:
# Load the cleaned data
df = joblib.load('../data/cleaned_data.pkl')

# Map target variable: yes -> 1, no -> 0
df['personal_loan'] = df['personal_loan'].map({'yes': 1, 'no': 0})

# Separate features and target
X = df.drop(['personal_loan', 'customer_id'], axis=1)
y = df['personal_loan']

print(f"Data loaded. X shape: {X.shape}, y shape: {y.shape}")

## 2. Train/Test Split (STRICT LEAKAGE PREVENTION)
We **MUST** split the data first before any transformations to prevent data leakage. This ensures the model only learns patterns from the training set.

In [ ]:
# Perform the 80/20 train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Make a copy to avoid SettingWithCopyWarning
X_train = X_train.copy()

print(f"Train set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

## 3. One-Hot Encoding (Post-Split Transformation)
After the split, we identify all categorical columns and encode them into integers. This is done separately on training and test sets to prevent leakage.

In [ ]:
# Identify ALL categorical columns
categorical_cols = ['education_level', 'credit_card_acct', 'online_acct']

# Encode categorical variables in training set
for col in categorical_cols:
    dummies_train = pd.get_dummies(X_train[col], prefix=col, dtype=int)
    X_train = pd.concat([X_train, dummies_train], axis=1).drop([col], axis=1)

# Align test set to match training set columns
X_test, X_train = X_test.align(X_train, join='right', axis=1, fill_value=0)

print(f"Encoding complete. X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

## 4. Feature Selection (SelectKBest with Chi-Square)
We apply `SelectKBest` with chi-square scoring to identify the **k=7** most predictive features. This MUST come BEFORE standardisation because chi-square requires non-negative values.

In [ ]:
# Apply SelectKBest to identify k=7 best features
selector = SelectKBest(score_func=chi2, k=7)
X_train_selected_array = selector.fit_transform(X_train, y_train)
X_test_selected_array = selector.transform(X_test)

# Get names of selected features
selected_features = X_train.columns[selector.get_support()]
print(f"Selected Features (k=7): {list(selected_features)}")

# Convert back to DataFrame
X_train_selected = pd.DataFrame(X_train_selected_array, columns=selected_features, index=X_train.index)
X_test_selected = pd.DataFrame(X_test_selected_array, columns=selected_features, index=X_test.index)

print(f"X_train_selected shape: {X_train_selected.shape}, X_test_selected shape: {X_test_selected.shape}")

## 5. Standardisation (Post-Feature Selection)
We apply `StandardScaler` to continuous features ONLY within our selected feature set. This normalises values to mean=0, std=1 for better model training.

In [ ]:
# Define continuous features from original dataset
continuous_features = ['age', 'credit_amt_used', 'credit_limit', 'family_size', 'income', 'months_customer']

# Filter to only those in selected features
continuous_selected = [col for col in continuous_features if col in X_train_selected.columns]

# Apply StandardScaler to continuous features only
for col in continuous_selected:
    scaler = StandardScaler()
    X_train_selected[col] = scaler.fit_transform(X_train_selected[[col]].values)
    X_test_selected[col] = scaler.transform(X_test_selected[[col]].values)

print(f"Standardisation complete. Scaled features: {continuous_selected}")

## 6. Train Baseline Random Forest Classifier
We train a baseline Random Forest with default parameters on the selected, standardised feature set.

In [ ]:
# Train baseline Random Forest Classifier
rf_model_selected = RandomForestClassifier(random_state=42)
rf_model_selected.fit(X_train_selected, y_train)

# Predictions
y_pred = rf_model_selected.predict(X_test_selected)

print("Random Forest (Selected Features k=7) - Baseline Model trained.")

## 7. Evaluate Model Performance
We calculate F1-Score (primary metric for imbalanced classes), precision, recall, and the confusion matrix.

In [ ]:
# Calculate evaluation metrics
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("=== RF (Selected Features k=7) - Baseline Performance ===")
print(f"F1-Score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"\nConfusion Matrix:\n{cm}")